# División estratégica del dataset de mora: 70 % / 15 % / 15 %

Este notebook recibe `base_mora_ecuador_21500_final.csv` y genera tres conjuntos independientes. La división se realiza dentro de cada uno de los 24 meses y se estratifica por la variable textual `mora`, garantizando representación temporal y de todas las clases objetivo.

Todas las líneas ejecutables incluyen su comentario explicativo en español.


## 1. Configuración


In [1]:
# Línea 0001: Importa el recurso específico necesario para esta etapa del proceso.
from pathlib import Path
# Línea 0002: Importa la librería necesaria para ejecutar esta operación.
import pandas as pd
# Línea 0003: Importa el recurso específico necesario para esta etapa del proceso.
from sklearn.model_selection import train_test_split

# Línea 0004: Calcula o asigna `SEMILLA` para utilizarlo en los pasos siguientes.
SEMILLA = 20260925
# Línea 0005: Calcula o asigna `ARCHIVO_BASE` para utilizarlo en los pasos siguientes.
ARCHIVO_BASE = Path("base_mora_ecuador_21500_final.csv")
# Línea 0006: Calcula o asigna `ARCHIVO_TRAIN` para utilizarlo en los pasos siguientes.
ARCHIVO_TRAIN = Path("data_train-mora_final.csv")
# Línea 0007: Calcula o asigna `ARCHIVO_VALIDATION` para utilizarlo en los pasos siguientes.
ARCHIVO_VALIDATION = Path("data_validation-mora_final.csv")
# Línea 0008: Calcula o asigna `ARCHIVO_TEST` para utilizarlo en los pasos siguientes.
ARCHIVO_TEST = Path("data_test-mora_final.csv")
# Línea 0009: Calcula o asigna `TOTAL_ESPERADO` para utilizarlo en los pasos siguientes.
TOTAL_ESPERADO = 21500
# Línea 0010: Calcula o asigna `FILAS_TRAIN` para utilizarlo en los pasos siguientes.
FILAS_TRAIN = 15050
# Línea 0011: Calcula o asigna `FILAS_VALIDATION` para utilizarlo en los pasos siguientes.
FILAS_VALIDATION = 3225
# Línea 0012: Calcula o asigna `FILAS_TEST` para utilizarlo en los pasos siguientes.
FILAS_TEST = 3225
# Línea 0013: Calcula o asigna `COLUMNA_FECHA` para utilizarlo en los pasos siguientes.
COLUMNA_FECHA = "fecha_reporte"
# Línea 0014: Calcula o asigna `COLUMNA_OBJETIVO` para utilizarlo en los pasos siguientes.
COLUMNA_OBJETIVO = "mora"
# Línea 0015: Calcula o asigna `COLUMNAS_CATEGORICAS` para utilizarlo en los pasos siguientes.
COLUMNAS_CATEGORICAS = ["mora", "zona", "tipo_credito", "vivienda", "nivel_ahorro", "clasif_sbs", "nivel_educ", "reestructuraciones_refinanciamientos_previos"]
# Línea 0016: Calcula o asigna `COLUMNAS_ENTERAS` para utilizarlo en los pasos siguientes.
COLUMNAS_ENTERAS = ["atraso", "edad", "dias_lab", "exp_sf", "score", "mora_max_3m", "mora_max_6m", "mora_max_12m", "atrasos_6m_12m", "atrasos_12m", "num_entidades_deuda", "nuevos_creditos_3m_6m", "nuevos_creditos_6m", "cuotas_restantes", "antiguedad_cliente_banco", "numero_productos_banco"]


## 2. Carga y validación de la base completa


In [2]:
# Línea 0001: Evalúa esta condición antes de ejecutar el bloque asociado.
if not ARCHIVO_BASE.exists():
    # Línea 0002: Inicia un bloque controlado para manejar diferencias entre Colab y el entorno local.
    try:
        # Línea 0003: Importa el recurso específico necesario para esta etapa del proceso.
        from google.colab import files
    # Línea 0004: Controla el caso en que Google Colab no esté disponible.
    except ImportError as error:
        # Línea 0005: Detiene el proceso y comunica la validación que no fue superada.
        raise FileNotFoundError(f"No se encontró {ARCHIVO_BASE.name} junto al notebook.") from error
    # Línea 0006: Calcula o asigna `archivos_subidos` para utilizarlo en los pasos siguientes.
    archivos_subidos = files.upload()
    # Línea 0007: Evalúa esta condición antes de ejecutar el bloque asociado.
    if ARCHIVO_BASE.name not in archivos_subidos:
        # Línea 0008: Detiene el proceso y comunica la validación que no fue superada.
        raise FileNotFoundError(f"Debe cargar el archivo {ARCHIVO_BASE.name}.")

# Línea 0009: Calcula o asigna `datos` para utilizarlo en los pasos siguientes.
datos = pd.read_csv(ARCHIVO_BASE, dtype={"ID": "string", COLUMNA_FECHA: "string"})
# Línea 0010: Recorre los elementos requeridos para aplicar la operación de forma repetida.
for columna in COLUMNAS_ENTERAS:
    # Línea 0011: Calcula o asigna `datos[columna]` para utilizarlo en los pasos siguientes.
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce").astype("Int64")
# Línea 0012: Evalúa esta condición antes de ejecutar el bloque asociado.
if datos.shape != (TOTAL_ESPERADO, 35):
    # Línea 0013: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError(f"La base tiene dimensiones {datos.shape} y se esperaban (21500, 35).")
# Línea 0014: Evalúa esta condición antes de ejecutar el bloque asociado.
if datos.columns[0] != "ID":
    # Línea 0015: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError("ID debe ser la primera columna.")
# Línea 0016: Evalúa esta condición antes de ejecutar el bloque asociado.
if datos.columns[1] != COLUMNA_FECHA:
    # Línea 0017: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError("fecha_reporte debe ser la segunda columna.")
# Línea 0018: Evalúa esta condición antes de ejecutar el bloque asociado.
if datos.columns[-1] != COLUMNA_OBJETIVO:
    # Línea 0019: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError("mora debe ser la última columna.")
# Línea 0020: Evalúa esta condición antes de ejecutar el bloque asociado.
if not datos["ID"].str.fullmatch(r"\d{7}XXX").all() or not datos["ID"].is_unique:
    # Línea 0021: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError("Los identificadores deben estar protegidos y ser únicos.")
# Línea 0022: Evalúa esta condición antes de ejecutar el bloque asociado.
if datos[COLUMNA_OBJETIVO].isna().any():
    # Línea 0023: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError("La columna mora no puede contener valores faltantes.")

# Línea 0024: Calcula o asigna `fechas_mensuales` para utilizarlo en los pasos siguientes.
fechas_mensuales = sorted(datos[COLUMNA_FECHA].dropna().unique().tolist())
# Línea 0025: Evalúa esta condición antes de ejecutar el bloque asociado.
if len(fechas_mensuales) != 24:
    # Línea 0026: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError("La base debe contener exactamente 24 fechas mensuales.")
# Línea 0027: Calcula o asigna `conteos_base` para utilizarlo en los pasos siguientes.
conteos_base = datos[COLUMNA_FECHA].value_counts()
# Línea 0028: Evalúa esta condición antes de ejecutar el bloque asociado.
if conteos_base.max() - conteos_base.min() > 1:
    # Línea 0029: Detiene el proceso y comunica la validación que no fue superada.
    raise ValueError("Las fechas de la base no están distribuidas de forma equilibrada.")
# Línea 0030: Muestra este resultado para facilitar la revisión del proceso.
print(f"Base validada: {datos.shape[0]} filas, {datos.shape[1]} columnas y {len(fechas_mensuales)} meses.")


Base validada: 21500 filas, 35 columnas y 24 meses.


## 3. División mensual estratificada por mora


In [3]:
# Línea 0001: Calcula o asigna `partes_train` para utilizarlo en los pasos siguientes.
partes_train = []
# Línea 0002: Calcula o asigna `partes_validation` para utilizarlo en los pasos siguientes.
partes_validation = []
# Línea 0003: Calcula o asigna `partes_test` para utilizarlo en los pasos siguientes.
partes_test = []

# Línea 0004: Recorre los elementos requeridos para aplicar la operación de forma repetida.
for posicion, fecha in enumerate(fechas_mensuales):
    # Línea 0005: Calcula o asigna `datos_mes` para utilizarlo en los pasos siguientes.
    datos_mes = datos.loc[datos[COLUMNA_FECHA].eq(fecha)].copy()
    # Línea 0006: Calcula o asigna `cuota_train` para utilizarlo en los pasos siguientes.
    cuota_train = 627 + int(posicion < 2)
    # Línea 0007: Ejecuta esta instrucción como parte del procedimiento descrito.
    cuota_validation = 134 + int(2 <= posicion < 11)
    # Línea 0008: Ejecuta esta instrucción como parte del procedimiento descrito.
    cuota_test = 134 + int(11 <= posicion < 20)
    # Línea 0009: Evalúa esta condición antes de ejecutar el bloque asociado.
    if cuota_train + cuota_validation + cuota_test != len(datos_mes):
        # Línea 0010: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"Las cuotas no coinciden con el número de registros del mes {fecha}.")
    # Línea 0011: Calcula o asigna `train_mes, temporal_mes` para utilizarlo en los pasos siguientes.
    train_mes, temporal_mes = train_test_split(datos_mes, train_size=cuota_train, test_size=cuota_validation + cuota_test, random_state=SEMILLA + posicion, shuffle=True, stratify=datos_mes[COLUMNA_OBJETIVO])
    # Línea 0012: Calcula o asigna `validation_mes, test_mes` para utilizarlo en los pasos siguientes.
    validation_mes, test_mes = train_test_split(temporal_mes, train_size=cuota_validation, test_size=cuota_test, random_state=SEMILLA + 100 + posicion, shuffle=True, stratify=temporal_mes[COLUMNA_OBJETIVO])
    # Línea 0013: Ejecuta esta instrucción como parte del procedimiento descrito.
    partes_train.append(train_mes)
    # Línea 0014: Ejecuta esta instrucción como parte del procedimiento descrito.
    partes_validation.append(validation_mes)
    # Línea 0015: Ejecuta esta instrucción como parte del procedimiento descrito.
    partes_test.append(test_mes)

# Línea 0016: Calcula o asigna `data_train` para utilizarlo en los pasos siguientes.
data_train = pd.concat(partes_train, ignore_index=True).sample(frac=1, random_state=SEMILLA + 200).reset_index(drop=True)
# Línea 0017: Calcula o asigna `data_validation` para utilizarlo en los pasos siguientes.
data_validation = pd.concat(partes_validation, ignore_index=True).sample(frac=1, random_state=SEMILLA + 201).reset_index(drop=True)
# Línea 0018: Calcula o asigna `data_test` para utilizarlo en los pasos siguientes.
data_test = pd.concat(partes_test, ignore_index=True).sample(frac=1, random_state=SEMILLA + 202).reset_index(drop=True)

# Línea 0019: Evalúa esta condición antes de ejecutar el bloque asociado.
if len(data_train) != FILAS_TRAIN:
    # Línea 0020: Detiene el proceso y comunica la validación que no fue superada.
    raise AssertionError(f"Train contiene {len(data_train)} filas y se esperaban {FILAS_TRAIN}.")
# Línea 0021: Evalúa esta condición antes de ejecutar el bloque asociado.
if len(data_validation) != FILAS_VALIDATION:
    # Línea 0022: Detiene el proceso y comunica la validación que no fue superada.
    raise AssertionError(f"Validation contiene {len(data_validation)} filas y se esperaban {FILAS_VALIDATION}.")
# Línea 0023: Evalúa esta condición antes de ejecutar el bloque asociado.
if len(data_test) != FILAS_TEST:
    # Línea 0024: Detiene el proceso y comunica la validación que no fue superada.
    raise AssertionError(f"Test contiene {len(data_test)} filas y se esperaban {FILAS_TEST}.")
# Línea 0025: Muestra este resultado para facilitar la revisión del proceso.
print(f"División creada: train={len(data_train)}, validation={len(data_validation)}, test={len(data_test)}")


División creada: train=15050, validation=3225, test=3225


## 4. Validaciones de cobertura y ausencia de fuga


In [4]:
# Línea 0001: Define la función `validar_atraso_mora` para reutilizar este procedimiento de forma ordenada.
def validar_atraso_mora(conjunto):
    # Línea 0002: Calcula o asigna `rangos` para utilizarlo en los pasos siguientes.
    rangos = {"AL DÍA": (0, 0), "MORA DE 1 A 7 DÍAS": (1, 7), "MORA DE 8 A 30 DÍAS": (8, 30), "MORA DE 31 A 60 DÍAS": (31, 60), "MORA DE 61 A 90 DÍAS": (61, 90), "MORA DE MÁS DE 90 DÍAS": (91, float("inf"))}
    # Línea 0003: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for clase, limites in rangos.items():
        # Línea 0004: Calcula o asigna `atrasos` para utilizarlo en los pasos siguientes.
        atrasos = conjunto.loc[conjunto[COLUMNA_OBJETIVO].eq(clase), "atraso"].dropna()
        # Línea 0005: Evalúa esta condición antes de ejecutar el bloque asociado.
        if not atrasos.between(limites[0], limites[1]).all():
            # Línea 0006: Devuelve el resultado calculado por la función.
            return False
    # Línea 0007: Devuelve el resultado calculado por la función.
    return True


# Línea 0008: Define la función `validar_conjunto` para reutilizar este procedimiento de forma ordenada.
def validar_conjunto(nombre, conjunto, base):
    # Línea 0009: Evalúa esta condición antes de ejecutar el bloque asociado.
    if list(conjunto.columns) != list(base.columns):
        # Línea 0010: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"{nombre} no conserva el esquema original.")
    # Línea 0011: Evalúa esta condición antes de ejecutar el bloque asociado.
    if conjunto.columns[0] != "ID" or conjunto.columns[1] != COLUMNA_FECHA or conjunto.columns[-1] != COLUMNA_OBJETIVO:
        # Línea 0012: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"{nombre} no conserva el orden obligatorio de las columnas.")
    # Línea 0013: Evalúa esta condición antes de ejecutar el bloque asociado.
    if set(conjunto[COLUMNA_FECHA]) != set(fechas_mensuales):
        # Línea 0014: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"{nombre} no contiene las 24 fechas mensuales.")
    # Línea 0015: Evalúa esta condición antes de ejecutar el bloque asociado.
    if conjunto.groupby([COLUMNA_FECHA, COLUMNA_OBJETIVO]).ngroups != 144:
        # Línea 0016: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"{nombre} no contiene las 144 combinaciones fecha-mora.")
    # Línea 0017: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for columna in COLUMNAS_CATEGORICAS:
        # Línea 0018: Evalúa esta condición antes de ejecutar el bloque asociado.
        if set(conjunto[columna].dropna().unique()) != set(base[columna].dropna().unique()):
            # Línea 0019: Detiene el proceso y comunica la validación que no fue superada.
            raise AssertionError(f"{nombre} no conserva todas las categorías de {columna}.")
    # Línea 0020: Calcula o asigna `columnas_con_faltantes` para utilizarlo en los pasos siguientes.
    columnas_con_faltantes = base.columns[base.isna().any()].tolist()
    # Línea 0021: Calcula o asigna `faltantes_ausentes` para utilizarlo en los pasos siguientes.
    faltantes_ausentes = [columna for columna in columnas_con_faltantes if not conjunto[columna].isna().any()]
    # Línea 0022: Evalúa esta condición antes de ejecutar el bloque asociado.
    if faltantes_ausentes:
        # Línea 0023: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"{nombre} no representa faltantes de: {faltantes_ausentes}")
    # Línea 0024: Evalúa esta condición antes de ejecutar el bloque asociado.
    if not validar_atraso_mora(conjunto):
        # Línea 0025: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"{nombre} contiene inconsistencias entre atraso y mora.")
    # Línea 0026: Calcula o asigna `conteos_mensuales` para utilizarlo en los pasos siguientes.
    conteos_mensuales = conjunto[COLUMNA_FECHA].value_counts()
    # Línea 0027: Evalúa esta condición antes de ejecutar el bloque asociado.
    if conteos_mensuales.max() - conteos_mensuales.min() > 1:
        # Línea 0028: Detiene el proceso y comunica la validación que no fue superada.
        raise AssertionError(f"{nombre} no conserva una distribución mensual equilibrada.")
    # Línea 0029: Devuelve el resultado calculado por la función.
    return f"{nombre}: validación aprobada"


# Línea 0030: Muestra este resultado para facilitar la revisión del proceso.
print(validar_conjunto("TRAIN", data_train, datos))
# Línea 0031: Muestra este resultado para facilitar la revisión del proceso.
print(validar_conjunto("VALIDATION", data_validation, datos))
# Línea 0032: Muestra este resultado para facilitar la revisión del proceso.
print(validar_conjunto("TEST", data_test, datos))

# Línea 0033: Calcula o asigna `ids_train` para utilizarlo en los pasos siguientes.
ids_train = set(data_train["ID"])
# Línea 0034: Calcula o asigna `ids_validation` para utilizarlo en los pasos siguientes.
ids_validation = set(data_validation["ID"])
# Línea 0035: Calcula o asigna `ids_test` para utilizarlo en los pasos siguientes.
ids_test = set(data_test["ID"])
# Línea 0036: Evalúa esta condición antes de ejecutar el bloque asociado.
if (ids_train & ids_validation) or (ids_train & ids_test) or (ids_validation & ids_test):
    # Línea 0037: Detiene el proceso y comunica la validación que no fue superada.
    raise AssertionError("Existen IDs compartidos entre los conjuntos.")
# Línea 0038: Evalúa esta condición antes de ejecutar el bloque asociado.
if ids_train | ids_validation | ids_test != set(datos["ID"]):
    # Línea 0039: Detiene el proceso y comunica la validación que no fue superada.
    raise AssertionError("La unión de los conjuntos no reconstruye la base completa.")

# Línea 0040: Calcula o asigna `distribucion_base` para utilizarlo en los pasos siguientes.
distribucion_base = datos[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index()
# Línea 0041: Calcula o asigna `comparacion_mora` para utilizarlo en los pasos siguientes.
comparacion_mora = pd.DataFrame({"base_pct": distribucion_base, "train_pct": data_train[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index(), "validation_pct": data_validation[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index(), "test_pct": data_test[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index()}).round(4)
# Línea 0042: Calcula o asigna `desviacion_maxima` para utilizarlo en los pasos siguientes.
desviacion_maxima = comparacion_mora[["train_pct", "validation_pct", "test_pct"]].sub(comparacion_mora["base_pct"], axis=0).abs().to_numpy().max()
# Línea 0043: Evalúa esta condición antes de ejecutar el bloque asociado.
if desviacion_maxima > 0.10:
    # Línea 0044: Detiene el proceso y comunica la validación que no fue superada.
    raise AssertionError(f"La desviación máxima de mora es {desviacion_maxima:.4f} puntos porcentuales.")
# Línea 0045: Muestra este resultado para facilitar la revisión del proceso.
print(comparacion_mora.to_string())
# Línea 0046: Muestra este resultado para facilitar la revisión del proceso.
print(f"Desviación máxima de mora: {desviacion_maxima:.4f} puntos porcentuales.")


TRAIN: validación aprobada
VALIDATION: validación aprobada
TEST: validación aprobada
                        base_pct  train_pct  validation_pct  test_pct
mora                                                                 
AL DÍA                   72.0047    72.0066         72.0000   72.0000
MORA DE 1 A 7 DÍAS       10.0000     9.9867         10.0155   10.0465
MORA DE 31 A 60 DÍAS      4.4977     4.4983          4.4961    4.4961
MORA DE 61 A 90 DÍAS      3.4977     3.4950          3.5349    3.4729
MORA DE 8 A 30 DÍAS       7.0000     7.0033          7.0078    6.9767
MORA DE MÁS DE 90 DÍAS    3.0000     3.0100          2.9457    3.0078
Desviación máxima de mora: 0.0543 puntos porcentuales.


## 5. Exportación de los tres conjuntos


In [5]:
# Línea 0001: Exporta el conjunto en formato CSV sin agregar el índice interno de pandas.
data_train.to_csv(ARCHIVO_TRAIN, index=False, encoding="utf-8-sig")
# Línea 0002: Exporta el conjunto en formato CSV sin agregar el índice interno de pandas.
data_validation.to_csv(ARCHIVO_VALIDATION, index=False, encoding="utf-8-sig")
# Línea 0003: Exporta el conjunto en formato CSV sin agregar el índice interno de pandas.
data_test.to_csv(ARCHIVO_TEST, index=False, encoding="utf-8-sig")
# Línea 0004: Muestra este resultado para facilitar la revisión del proceso.
print(f"Creado: {ARCHIVO_TRAIN.name} | {len(data_train)} filas")
# Línea 0005: Muestra este resultado para facilitar la revisión del proceso.
print(f"Creado: {ARCHIVO_VALIDATION.name} | {len(data_validation)} filas")
# Línea 0006: Muestra este resultado para facilitar la revisión del proceso.
print(f"Creado: {ARCHIVO_TEST.name} | {len(data_test)} filas")


Creado: data_train-mora_final.csv | 15050 filas
Creado: data_validation-mora_final.csv | 3225 filas
Creado: data_test-mora_final.csv | 3225 filas


## 6. Descarga en Google Colab


In [6]:
# Línea 0001: Inicia un bloque controlado para manejar diferencias entre Colab y el entorno local.
try:
    # Línea 0002: Importa el recurso específico necesario para esta etapa del proceso.
    from google.colab import files
# Línea 0003: Controla el caso en que Google Colab no esté disponible.
except ImportError:
    # Línea 0004: Muestra este resultado para facilitar la revisión del proceso.
    print("Ejecución local: los tres CSV quedaron guardados junto al notebook.")
# Línea 0005: Ejecuta la alternativa cuando la condición previa no se cumple.
else:
    # Línea 0006: Inicia la descarga del archivo generado cuando se trabaja en Google Colab.
    files.download(str(ARCHIVO_TRAIN))
    # Línea 0007: Inicia la descarga del archivo generado cuando se trabaja en Google Colab.
    files.download(str(ARCHIVO_VALIDATION))
    # Línea 0008: Inicia la descarga del archivo generado cuando se trabaja en Google Colab.
    files.download(str(ARCHIVO_TEST))


Ejecución local: los tres CSV quedaron guardados junto al notebook.
